<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L59_Custom_Training_Frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L59: Custom Training Frameworks - Modular Training and MLOps

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 59 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Structure a minimal training framework: config, trainer, logging, checkpointing
- Add experiment tracking (e.g. log metrics to a dict or file)
- Design a reusable training loop that supports different models and tasks

---

## Concept: Custom Frameworks

**Modular training**: separate config (YAML/JSON), data loading, model build, training loop, and logging. **Experiment tracking**: log loss, LR, metrics to W&B, MLflow, or CSV. **MLOps**: CI for training, model registry, and deployment. We build a minimal trainer class and config.

---

In [ ]:
!pip install transformers torch -q
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import json
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Config

---

In [ ]:
config = {
    "model_name": "distilbert-base-uncased",
    "num_labels": 2,
    "lr": 2e-5,
    "batch_size": 8,
    "epochs": 2,
    "output_dir": "./out_framework_l59",
}
with open("config_l59.json", "w") as f:
    json.dump(config, f, indent=2)
print("Config saved. Load with json.load for reproducibility.")

## Part 2: Minimal Trainer Class

---

In [ ]:
class MinimalTrainer:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])
        self.logs = []

    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0.0
        for batch in loader:
            loss = self.model(**batch).loss
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()
            total_loss += loss.item()
        return total_loss / len(loader)

    def train(self, loader):
        for epoch in range(self.config["epochs"]):
            avg_loss = self.train_epoch(loader)
            self.logs.append({"epoch": epoch, "loss": avg_loss})
            print(f"Epoch {epoch} loss: {avg_loss:.4f}")

trainer = MinimalTrainer(None, config)
print("Trainer: config-driven, logs list for simple tracking.")

## Part 3: Checkpointing and Logging

---

In [ ]:
# In train(): torch.save(model.state_dict(), f"{output_dir}/ckpt_epoch{epoch}.pt")
# Log to W&B: wandb.log({"loss": avg_loss})
print("Add checkpointing every N steps; log to wandb/mlflow for full MLOps.")

## Exercises

1. Add validation loop and early stopping to MinimalTrainer.
2. Load config from JSON and run training; save logs to CSV.
3. Integrate one experiment tracker (e.g. wandb.init, wandb.log).

---

## Key Takeaways

1. Config + trainer + logging = minimal framework; extend with eval and checkpointing.
2. Experiment tracking (W&B, MLflow) improves reproducibility and comparison.
3. MLOps: version configs, register models, automate training pipelines.

---

## Next Lesson

**L60: Capstone Project** – End-to-end LLM application.

---